# Part 2: RFM Segmentation & Retention Strategy
**D2C Customer Churn Intelligence Project** | Snapshot: `2025-09-30`

## 0. Imports & Config

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

DATA_DIR = Path('data')   # place all 7 CSVs here
SNAPSHOT = pd.Timestamp('2025-09-30')


## 1. Load Data

In [ ]:
customers     = pd.read_csv(DATA_DIR/'customers.csv')
orders        = pd.read_csv(DATA_DIR/'orders.csv',               parse_dates=['order_date'])
tickets       = pd.read_csv(DATA_DIR/'support_tickets.csv',      parse_dates=['ticket_date'])
web           = pd.read_csv(DATA_DIR/'web_events_snapshot.csv')
interventions = pd.read_csv(DATA_DIR/'intervention_history.csv')
labels        = pd.read_csv(DATA_DIR/'churn_labels.csv')

# Pre-snapshot only + remove _DUP rows
pre_orders = orders[orders.order_date <= SNAPSHOT].copy()
pre_orders['order_id_base'] = pre_orders['order_id'].str.replace('_DUP','', regex=False)
pre_orders = pre_orders.sort_values('order_id').drop_duplicates('order_id_base', keep='first')

print(f"Pre-snapshot clean orders: {len(pre_orders):,}")


## 2. Build RFM Features

In [ ]:
rfm = pre_orders.groupby('customer_id').agg(
    last_order_date  = ('order_date',    'max'),
    recency_days     = ('order_date',    lambda s: (SNAPSHOT - s.max()).days),
    frequency        = ('order_id_base', 'count'),
    monetary         = ('gross_amount',  'sum'),
    avg_discount_pct = ('discount_pct',  'mean'),
    full_price_rate  = ('discount_pct',  lambda s: (s == 0).mean()),
    return_rate      = ('returned',      'mean'),
    category_diversity = ('category',    'nunique')
).reset_index()

# All 2,400 customers (fill those with no orders)
rfm_full = customers[['customer_id']].merge(rfm, on='customer_id', how='left')
rfm_full['recency_days']      = rfm_full['recency_days'].fillna(999).astype(int)
rfm_full['frequency']         = rfm_full['frequency'].fillna(0).astype(int)
rfm_full['monetary']          = rfm_full['monetary'].fillna(0)
rfm_full['avg_discount_pct']  = rfm_full['avg_discount_pct'].fillna(0)
rfm_full['full_price_rate']   = rfm_full['full_price_rate'].fillna(0)
rfm_full['return_rate']       = rfm_full['return_rate'].fillna(0)
rfm_full['category_diversity']= rfm_full['category_diversity'].fillna(0).astype(int)

print(rfm_full[['recency_days','frequency','monetary']].describe().round(1))


## 3. Add Non-RFM Behavioral Signals

In [ ]:
# Signal 1: Support tickets (90-day window)
ticket_agg = (
    tickets[
        (tickets.ticket_date >= SNAPSHOT - pd.Timedelta(days=90)) &
        (tickets.ticket_date <= SNAPSHOT)
    ]
    .groupby('customer_id')
    .agg(
        ticket_count_90d         = ('ticket_id',       'count'),
        negative_ticket_rate_90d = ('sentiment_score', lambda s: (s < 0).mean())
    )
    .reset_index()
)

# Signal 2: Web/app activity (30-day snapshot)
web_signals = web[['customer_id','sessions_30d','abandoned_carts_30d','last_visit_days_ago']]

rfm_full = (rfm_full
    .merge(ticket_agg,  on='customer_id', how='left')
    .merge(web_signals, on='customer_id', how='left')
    .merge(labels[['customer_id','churn_next_60d']], on='customer_id', how='left')
    .merge(interventions[['customer_id','last_campaign_cost','manual_priority_bucket']],
           on='customer_id', how='left')
)

rfm_full['ticket_count_90d']        = rfm_full['ticket_count_90d'].fillna(0)
rfm_full['negative_ticket_rate_90d']= rfm_full['negative_ticket_rate_90d'].fillna(0)
rfm_full['sessions_30d']            = rfm_full['sessions_30d'].fillna(0)
rfm_full['abandoned_carts_30d']     = rfm_full['abandoned_carts_30d'].fillna(0)
rfm_full['last_visit_days_ago']     = rfm_full['last_visit_days_ago'].fillna(999)

print(f"Feature table: {rfm_full.shape}")


## 4. Segmentation Logic

Segments are assigned in priority order (first match wins):

| Segment | Rule |
|---|---|
| **Champions** | recency ≤ 30d, frequency ≥ 5, monetary ≥ ₹4,000 |
| **Loyal but at risk** | frequency ≥ 4, recency > 90d |
| **Support-stressed** | ticket_count_90d ≥ 1, negative_ticket_rate > 0.5 |
| **Discount hunters** | avg_discount_pct ≥ 0.30 |
| **Dormant** | recency > 180d, sessions_30d = 0 |
| **Maintain** | All others |


In [ ]:
def assign_segment(row):
    if row['recency_days'] <= 30 and row['frequency'] >= 5 and row['monetary'] >= 4000:
        return 'Champions'
    elif row['frequency'] >= 4 and row['recency_days'] > 90:
        return 'Loyal but at risk'
    elif row['ticket_count_90d'] >= 1 and row['negative_ticket_rate_90d'] > 0.5:
        return 'Support-stressed'
    elif row['avg_discount_pct'] >= 0.30:
        return 'Discount hunters'
    elif row['recency_days'] > 180 and row['sessions_30d'] == 0:
        return 'Dormant'
    else:
        return 'Maintain'

rfm_full['segment_name'] = rfm_full.apply(assign_segment, axis=1)

print("Segment counts:")
print(rfm_full['segment_name'].value_counts())
print()
print("Churn rate by segment:")
print(rfm_full.groupby('segment_name')['churn_next_60d'].mean().sort_values(ascending=False).round(3))


## 5. Visualisations

### 5.1 Segment size and churn rate

In [ ]:
seg_order = (rfm_full.groupby('segment_name')['churn_next_60d']
             .mean().sort_values(ascending=False).index)
colors = {'Champions':'#4CAF50','Loyal but at risk':'#FF5722',
          'Support-stressed':'#FF9800','Discount hunters':'#5B9BD5',
          'Dormant':'#9E9E9E','Maintain':'#8BC34A'}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

counts = rfm_full['segment_name'].value_counts().reindex(seg_order)
bars = axes[0].bar(seg_order, counts.values,
                   color=[colors[s] for s in seg_order], edgecolor='white', width=0.6)
axes[0].bar_label(bars, fmt='%d', padding=4)
axes[0].set_title('Customers per segment')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=20)

churn_rates = rfm_full.groupby('segment_name')['churn_next_60d'].mean().reindex(seg_order)
bars2 = axes[1].bar(seg_order, churn_rates.values,
                    color=[colors[s] for s in seg_order], edgecolor='white', width=0.6)
axes[1].bar_label(bars2, fmt='{:.1%}', padding=4)
axes[1].set_title('Churn rate per segment')
axes[1].set_ylabel('Churn rate')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1))
axes[1].set_ylim(0, 1.1)
axes[1].tick_params(axis='x', rotation=20)

plt.suptitle('RFM segment overview', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('rfm_chart_01_segment_overview.png', bbox_inches='tight')
plt.show()
print('Saved: rfm_chart_01_segment_overview.png')


### 5.2 RFM profile by segment

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, label in zip(axes,
    ['recency_days','frequency','monetary'],
    ['Avg recency (days)','Avg frequency (orders)','Avg monetary (₹)']):
    vals = rfm_full.groupby('segment_name')[col].mean().reindex(seg_order)
    bars = ax.bar(seg_order, vals.values,
                  color=[colors[s] for s in seg_order], edgecolor='white', width=0.6)
    ax.bar_label(bars, fmt='{:.0f}', padding=3, fontsize=9)
    ax.set_title(label)
    ax.tick_params(axis='x', rotation=20)

plt.suptitle('RFM profile by segment', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('rfm_chart_02_rfm_profile.png', bbox_inches='tight')
plt.show()
print('Saved: rfm_chart_02_rfm_profile.png')


### 5.3 Behavioral signals by segment

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, label in zip(axes,
    ['sessions_30d','ticket_count_90d','avg_discount_pct'],
    ['Avg sessions (30d)','Avg tickets (90d)','Avg discount rate']):
    vals = rfm_full.groupby('segment_name')[col].mean().reindex(seg_order)
    bars = ax.bar(seg_order, vals.values,
                  color=[colors[s] for s in seg_order], edgecolor='white', width=0.6)
    ax.bar_label(bars, fmt='{:.2f}', padding=3, fontsize=9)
    ax.set_title(label)
    ax.tick_params(axis='x', rotation=20)
    if col == 'avg_discount_pct':
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(1))

plt.suptitle('Behavioral signals by segment', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('rfm_chart_03_behavioral_signals.png', bbox_inches='tight')
plt.show()
print('Saved: rfm_chart_03_behavioral_signals.png')


### 5.4 Segment vs CRM manual priority

In [ ]:
cross = pd.crosstab(rfm_full['segment_name'],
                    rfm_full['manual_priority_bucket'],
                    normalize='index').round(2)
fig, ax = plt.subplots(figsize=(9, 4))
cross.reindex(seg_order).plot.bar(ax=ax, stacked=True,
    color=['#F44336','#FFC107','#4CAF50'], edgecolor='white', width=0.6)
ax.set_title('CRM manual priority mix within each RFM segment')
ax.set_ylabel('Proportion')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1))
ax.tick_params(axis='x', rotation=20)
ax.legend(title='Manual priority', bbox_to_anchor=(1.01,1), loc='upper left')
plt.tight_layout()
plt.savefig('rfm_chart_04_vs_crm_priority.png', bbox_inches='tight')
plt.show()
print('Saved: rfm_chart_04_vs_crm_priority.png')


## 6. Export `segments.csv`

In [ ]:
segments_out = rfm_full[[
    'customer_id','segment_name',
    'recency_days','frequency','monetary',
    'avg_discount_pct','full_price_rate','return_rate','category_diversity',
    'ticket_count_90d','negative_ticket_rate_90d',
    'sessions_30d','abandoned_carts_30d','last_visit_days_ago',
    'last_campaign_cost','manual_priority_bucket','churn_next_60d'
]].copy()

segments_out.to_csv('segments.csv', index=False)
print(f"Saved segments.csv — {len(segments_out):,} rows, {segments_out.shape[1]} columns")

# Segment summary
summary = (rfm_full.groupby('segment_name')
    .agg(customers=('customer_id','count'),
         churn_rate=('churn_next_60d','mean'),
         avg_recency=('recency_days','mean'),
         avg_frequency=('frequency','mean'),
         avg_monetary=('monetary','mean'),
         avg_campaign_cost=('last_campaign_cost','mean'))
    .round(2)
    .sort_values('churn_rate', ascending=False)
    .reset_index()
)
summary.to_csv('segment_summary.csv', index=False)
print("Saved segment_summary.csv")
display(summary)


## 7. Budget Prioritization

Assume a campaign budget of **₹25,000** at ~₹25 per contact.

Priority is based on **revenue at risk** = churn_rate × avg_monetary × avg_frequency.
Higher churn rate on high-value customers = most urgent.


In [ ]:
BUDGET = 25000
COST   = 25

budget_df = rfm_full.groupby('segment_name').agg(
    count       = ('customer_id',    'count'),
    churn_rate  = ('churn_next_60d',  'mean'),
    avg_monetary= ('monetary',        'mean'),
    avg_freq    = ('frequency',       'mean'),
).round(2)

budget_df['revenue_at_risk'] = (
    budget_df['churn_rate'] * budget_df['avg_monetary'] * budget_df['avg_freq']
).round(0)

budget_df = budget_df.sort_values('revenue_at_risk', ascending=False)
budget_df['campaign_budget_inr'] = (
    (budget_df['count'] * COST).clip(upper=BUDGET)
)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(budget_df.index, budget_df['revenue_at_risk'],
              color=[colors[s] for s in budget_df.index], edgecolor='white', width=0.6)
ax.bar_label(bars, fmt='₹{:.0f}', padding=4, fontsize=9)
ax.set_title('Expected revenue at risk per customer by segment')
ax.set_ylabel('Revenue at risk (INR)')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig('rfm_chart_05_budget_priority.png', bbox_inches='tight')
plt.show()
print('Saved: rfm_chart_05_budget_priority.png')

print()
print(budget_df[['count','churn_rate','avg_monetary','revenue_at_risk','campaign_budget_inr']].to_string())


## 8. Manual Review Cases

In [ ]:
# Borderline customers: R/F scores in the middle or high tickets + still active
review = rfm_full[
    ((rfm_full['recency_days'].between(60,120)) & (rfm_full['frequency'].between(3,5))) |
    ((rfm_full['ticket_count_90d'] >= 2) & (rfm_full['recency_days'] <= 30))
].copy()

print(f"Borderline candidates: {len(review)}")
cols = ['customer_id','recency_days','frequency','monetary','segment_name',
        'churn_next_60d','ticket_count_90d','sessions_30d','avg_discount_pct']
review[cols].head(12).to_string(index=False)


Full reasoning for 10 specific customers is in `manual_review_cases.md`.